In [1]:
import sys
sys.path.append('..')
from run.run_gspa import calculate_wavelet_dictionary
from run.run_ae_default_config import run_ae
import leidenalg
import scanpy, phate, meld
import numpy as np
import scprep
import matplotlib.pyplot as plt
import pandas as pd
from utils import *
import sklearn
from sklearn.preprocessing import scale

In [2]:
markers = ["Mki67", 'Birc5', 'Pclaf', 'Top2a', 'Hist1h1b', 'Stmn1',
           "Tcf7", "Lef1", "Ccr7", "Sell",
           "Slamf6", 'Xcl1',
           "Il7r", "Malat1", "Cxcr3", "Ltb", "Gpr183",
           "Irf7", "Stat1", 'Isg20', 'Ifit1', 'Ifit3', 'Isg15',
           "Nkg7", "Ccl5", "Ly6c2", "Lgals1", "Prf1", "Klrg1", "Cx3cr1", "Klre1", "Zeb2", "Gzma",
           "Pdcd1", "Cd101", "Havcr2"]

In [3]:
acute = scanpy.read_h5ad('data/acute_tcells.h5ad')
chronic = scanpy.read_h5ad('data/chronic_tcells.h5ad')

In [4]:
adata = scanpy.concat((acute,chronic))
adata.obs['batch'] = adata.obs['batch'].astype('category')

## Generate GSPA+QR features

In [7]:
scanpy.pp.subsample(adata, n_obs=20000)

In [ ]:
phate_op = phate.PHATE(random_state=42, n_jobs=-1, knn=30)
adata.obsm['X_phate'] = phate_op.fit_transform(adata.to_df())

Calculating PHATE...
  Running PHATE on 20000 observations and 14152 variables.
  Calculating graph and diffusion operator...
    Calculating PCA...
    Calculated PCA in 9.82 seconds.
    Calculating KNN search...


In [10]:
G_without_regression = phate_op.graph.to_pygsp()

In [13]:
data, data_hvgs = scprep.select.highly_variable_genes(adata.to_df(), adata.var_names, percentile=90)
data_hvg = data[data_hvgs]
data_hvg = data_hvg / np.linalg.norm(data_hvg, axis=0)

In [14]:
uniform_signal = np.ones((1, G_without_regression.N))
uniform_signal = uniform_signal / np.linalg.norm(uniform_signal, axis=1).reshape(-1,1)

In [16]:
cell_dictionary, wavelet_sizes = calculate_wavelet_dictionary(G_without_regression, use_reduced=True, J=5)

Maximum scale: 5


100%|██████████| 3/3 [43:57<00:00, 879.07s/it]


In [17]:
results = {}

signals_projected = project(data_hvg.T, cell_dictionary)
signals_reduced = svd(signals_projected)
results['signal_embedding'] = run_ae(signals_reduced)
np.savez('results/GSPA_QR_without_regression.npz', signal_embedding=results['signal_embedding'])

Epoch 1/100
43/43 [==============================] - 1s 9ms/step - loss: 0.0049 - val_loss: 0.0047
Epoch 2/100
43/43 [==============================] - 0s 6ms/step - loss: 0.0033 - val_loss: 0.0028
Epoch 3/100
43/43 [==============================] - 0s 5ms/step - loss: 0.0021 - val_loss: 0.0022
Epoch 4/100
43/43 [==============================] - 0s 5ms/step - loss: 0.0017 - val_loss: 0.0019
Epoch 5/100
43/43 [==============================] - 0s 5ms/step - loss: 0.0015 - val_loss: 0.0018
Epoch 6/100
43/43 [==============================] - 0s 5ms/step - loss: 0.0015 - val_loss: 0.0017
Epoch 7/100
43/43 [==============================] - 0s 5ms/step - loss: 0.0013 - val_loss: 0.0015
Epoch 8/100
43/43 [==============================] - 0s 5ms/step - loss: 0.0012 - val_loss: 0.0015
Epoch 9/100
43/43 [==============================] - 0s 5ms/step - loss: 0.0012 - val_loss: 0.0015
Epoch 10/100
43/43 [==============================] - 0s 5ms/step - loss: 0.0012 - val_loss: 0.0015
Epoch 11/

In [18]:
uniform_projected = project(uniform_signal, cell_dictionary)
results['localization_score'] = calculate_localization(uniform_projected, signals_projected)
np.savez('./results/GSPA_QR_without_regression.npz', signal_embedding=results['signal_embedding'],
         localization_score=results['localization_score'], genes=data_hvgs.values)

## Regress out proliferation
https://nbviewer.org/github/scverse/scanpy_usage/blob/master/180209_cell_cycle/cell_cycle.ipynb

In [28]:
!if [ ! -f data/regev_lab_cell_cycle_genes.txt ]; then curl -o data/regev_lab_cell_cycle_genes.txt https://raw.githubusercontent.com/theislab/scanpy_usage/master/180209_cell_cycle/data/regev_lab_cell_cycle_genes.txt; fi

In [39]:
cell_cycle_genes = [x.strip() for x in open('./data/regev_lab_cell_cycle_genes.txt')]
cell_cycle_genes = [x.capitalize() for x in cell_cycle_genes]

print(len(cell_cycle_genes))

# Split into 2 lists
s_genes = cell_cycle_genes[:43]
g2m_genes = cell_cycle_genes[43:]

cell_cycle_genes = [x for x in cell_cycle_genes if x in adata.var_names]
print(len(cell_cycle_genes))

In [31]:
scanpy.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)

In [32]:
adata_regressed = scanpy.pp.regress_out(adata, ['S_score', 'G2M_score'], copy=True)
scanpy.pp.scale(adata_regressed)

In [33]:
phate_op = phate.PHATE(random_state=42, n_jobs=-1, knn=30)
adata_regressed.obsm['X_phate'] = phate_op.fit_transform(adata_regressed.to_df())

Calculating PHATE...
  Running PHATE on 44190 observations and 14152 variables.
  Calculating graph and diffusion operator...
    Calculating PCA...
    Calculated PCA in 21.14 seconds.
    Calculating KNN search...
    Calculated KNN search in 61.14 seconds.
    Calculating affinities...
    Calculated affinities in 7.63 seconds.
  Calculated graph and diffusion operator in 90.52 seconds.
  Calculating landmark operator...
    Calculating SVD...
    Calculated SVD in 23.65 seconds.
    Calculating KMeans...
    Calculated KMeans in 5.00 seconds.
  Calculated landmark operator in 31.14 seconds.
  Calculating optimal t...
    Automatically selected t = 33
  Calculated optimal t in 1.70 seconds.
  Calculating diffusion potential...
  Calculated diffusion potential in 0.46 seconds.
  Calculating metric MDS...
  Calculated metric MDS in 5.67 seconds.
Calculated PHATE in 129.52 seconds.


In [34]:
G_with_regression = phate_op.graph

In [ ]:
scprep.plot.scatter2d(adata_regressed.obsm['X_phate'], c=adata_regressed.obs['batch'], ticks=None, legend_loc=(1.05,0), 
                      filename='./figures/samples_tcells_regressed.png', dpi=200, title='Cd3e+ cells')

In [ ]:
meld_op = meld.MELD()
meld_op.graph = G_with_regression
all_sample_densities = meld_op.transform(adata_regressed.obs['batch'])
all_sample_likelihoods = meld.normalize_densities(all_sample_densities)
adata_regressed.obs[[f'{x}_likelihood' for x in all_sample_likelihoods.columns]] = all_sample_likelihoods.values

In [ ]:
fig, ax = plt.subplots(2,3, figsize=(13,8), dpi=200); ax=ax.flatten()

for i,condition in enumerate(adata_regressed.obs['batch'].cat.categories):
    scprep.plot.scatter2d(adata_regressed.obsm['X_phate'], c=adata_regressed.obs[f'{condition}_likelihood'],
                     title=condition, label_prefix='Cell PHATE', ticks=None, ax=ax[i],
                      cmap=meld.get_meld_cmap())
    
plt.tight_layout()
fig.savefig('./figures/conditions_regressed.png', dpi=200)

In [ ]:
fig, ax = plt.subplots(6,6, figsize=(24,24), dpi=200); ax=ax.flatten()

for i,marker in enumerate(markers):
    scprep.plot.scatter2d(adata_regressed.obsm['X_phate'], c=adata_regressed.to_df()[marker],
                     title=marker, label_prefix='Cell PHATE', ticks=None, ax=ax[i])
    
plt.tight_layout()
fig.savefig('./figures/markers_regressed.png', dpi=200)

## Generate GSPA+QR features

In [ ]:
G_with_regression = G_with_regression.to_pygsp()

In [ ]:
uniform_signal = np.ones((1, G_with_regression.N))
uniform_signal = uniform_signal / np.linalg.norm(uniform_signal, axis=1).reshape(-1,1)

In [ ]:
cell_dictionary, wavelet_sizes = calculate_wavelet_dictionary(G_with_regression, use_reduced=True)

In [ ]:
data_hvg, data_hvgs = scprep.select.highly_variable_genes(adata_regressed.to_df(), adata_regressed.var_names, percentile=90)
data_hvg = data[:, data_hvgs]
data_hvg = data_hvg / np.linalg.norm(data_hvg, axis=0)

In [ ]:
results = {}

signals_projected = project(data_hvg.T, cell_dictionary)
signals_reduced = svd(signals_projected)
results['signal_embedding'] = run_ae(signals_reduced)
np.savez('./results/GSPA_QR_with_regression.npz', signal_embedding=results['signal_embedding'])

In [ ]:
uniform_projected = project(uniform_signal, cell_dictionary)
results['localization_score'] = calculate_localization(uniform_projected, signals_projected)
np.savez('./results/GSPA_QR_with_regression.npz', signal_embedding=results['signal_embedding'],
         localization_score=results['localization_score'])